In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import re

# ==============================
# CONFIGURACIÓN
# ==============================

ruta = "/home/luciam/Documents/starry/Orion_yceb/dat_orion_sergio/sample_3/dat"
archivos = sorted(glob.glob(os.path.join(ruta, "*.dat")))

print(f"\nSe encontraron {len(archivos)} archivos\n")

# ==============================
# FUNCIONES
# ==============================

def extraer_objeto_sector(nombre):
    match = re.search(r"(TIC_\d+)_S_(\d+)_LC", nombre)
    if match:
        return match.group(1), int(match.group(2))
    return None, None


def cargar_y_normalizar(archivo):
    datos = np.loadtxt(archivo, comments="#")

    time = datos[:, 0]
    mag = datos[:, 1]

    # convertir magnitud → flujo
    flux = 10 ** (-mag / 2.5)

    # normalizar
    flux_norm = flux / np.nanmedian(flux)

    return time, flux_norm


# ==============================
# AGRUPAR POR OBJETO
# ==============================

objetos = {}

for archivo in archivos:
    nombre = os.path.basename(archivo)
    objeto, sector = extraer_objeto_sector(nombre)

    if objeto is None:
        continue

    objetos.setdefault(objeto, []).append((archivo, sector))

print(f"Objetos únicos: {len(objetos)}\n")

# ==============================
# EXPLORACIÓN VISUAL
# ==============================

for objeto, lista in objetos.items():

    lista = sorted(lista, key=lambda x: x[1])
    n = len(lista)

    fig, axs = plt.subplots(1, n, figsize=(5*n, 4), sharey=True)

    if n == 1:
        axs = [axs]

    print(f"Explorando {objeto}")

    for ax, (archivo, sector) in zip(axs, lista):

        try:
            time, flux_norm = cargar_y_normalizar(archivo)

            ax.plot(time, flux_norm, '.', markersize=1, alpha=0.6)

            ax.set_title(f"Sector {sector}")
            ax.set_xlabel("Tiempo [d]")
            ax.set_ylabel("Flujo normalizado")

        except Exception as e:
            print(f"Error en {archivo}: {e}")

    plt.suptitle(f"{objeto}")
    plt.tight_layout()
    plt.show()

    plt.close(fig)

print("\nExploración terminada ✨")


In [ ]:
import lightkurve as lk

search = lk.search_lightcurve("TIC 11300044", mission="TESS")
print(search)

for i in range(len(search)):
    lc = search[i].download()   # descarga solo ese sector
    print(f"Sector {i} descargado")
    
    lc.plot()
    
    del lc  # libera memoria